# RexNet150


In [1]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import cv2
import random 
import numpy as np
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
from pytorch_lightning.loggers import TensorBoardLogger
from torchmetrics import Accuracy, F1Score, AUROC
import timm
import torchvision.transforms as transforms
from PIL import Image
from typing import List

Определим класс датасета

In [24]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

class VideoFramesDataset(Dataset):
    def __init__(self, root_dir, split='train', frames_per_video=32, frame_size=224,
                 max_real=None, max_fake=None, seed=42):
        self.root_dir = Path(root_dir)
        self.split = split
        self.frames_per_video = frames_per_video
        self.frame_size = frame_size
        
        self.rng = random.Random(seed) if split != 'train' else random
        
        if split == 'train':
            self.base_dir = self.root_dir / 'trainframesfaces'
        elif split == 'val':
            self.base_dir = self.root_dir / 'valframesfaces'
        else:
            self.base_dir = self.root_dir / 'testframesfaces'
        
        self.fake_dir = self.base_dir / 'fake_frames_faces'
        self.real_dir = self.base_dir / 'real_frames_faces'
        
        fake_paths = sorted([f for f in self.fake_dir.iterdir() if f.is_dir()]) if self.fake_dir.exists() else []
        real_paths = sorted([f for f in self.real_dir.iterdir() if f.is_dir()]) if self.real_dir.exists() else []
            
        print(f"Found {len(fake_paths)} fake videos in {split}")
        print(f"Found {len(real_paths)} real videos in {split}")
        
        if max_real is not None:
            real_paths = self.rng.sample(real_paths, min(len(real_paths), max_real))
        if max_fake is not None:
            fake_paths = self.rng.sample(fake_paths, min(len(fake_paths), max_fake))
            
        print(f"После фильтрации: {len(real_paths)} real, {len(fake_paths)} fake")
        
        combined = list(zip(fake_paths + real_paths, [1]*len(fake_paths) + [0]*len(real_paths)))
        self.rng.shuffle(combined)
        self.video_paths, self.labels = zip(*combined)
        self.video_paths = list(self.video_paths)
        self.labels = list(self.labels)
        
        print(f" Total {len(self.video_paths)} videos for {split} split")
        print(f" Real (0): {self.labels.count(0)}")
        print(f" Fake (1): {self.labels.count(1)}")

        self.transform = transforms.Compose([
            transforms.Resize(self.frame_size),
            transforms.CenterCrop(self.frame_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
        ])

    def __len__(self):
        return len(self.video_paths)
    
    def load_frames(self, video_path):
        frames = []
        frame_files = sorted([f for f in video_path.glob('*.jpg')])
        
        if len(frame_files) > self.frames_per_video:
            indices = np.linspace(0, len(frame_files)-1, self.frames_per_video, dtype=int)
            frame_files = [frame_files[i] for i in indices]
         
        for frame_file in frame_files:
            try:
                img = Image.open(frame_file).convert('RGB')
            except Exception:
                img = Image.new('RGB', (self.frame_size, self.frame_size))
            frames.append(img)
        
        while len(frames) < self.frames_per_video:
            frames.append(frames[-1] if frames else Image.new('RGB', (self.frame_size, self.frame_size)))
        
        return frames
    
    def __getitem__(self, idx):
        video_path = self.video_paths[idx]
        label = self.labels[idx]
        frames = self.load_frames(video_path)
        
        transformed_frames = [self.transform(frame) for frame in frames]
        frames_tensor = torch.stack(transformed_frames)
        
        return frames_tensor, torch.tensor(label, dtype=torch.long)

Определим функцию для создания даталоадеров

In [20]:
def create_dataloaders(root_dir, batch_size=4, num_workers=2, frames_per_video=16, val_max_real=None, val_max_fake=200, test_max_real=None, test_max_fake=600):
    
    train_dataset = VideoFramesDataset(root_dir=root_dir, split='train', frames_per_video=frames_per_video)
    val_dataset = VideoFramesDataset(root_dir=root_dir, split='val', frames_per_video=frames_per_video, max_real=val_max_real, max_fake=val_max_fake)
    test_dataset = VideoFramesDataset(root_dir=root_dir, split='test', frames_per_video=frames_per_video, max_real=test_max_real, max_fake=test_max_fake)
    
    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, pin_memory=True, persistent_workers=num_workers > 0)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True, persistent_workers=num_workers > 0)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True, persistent_workers=num_workers > 0)
    
    return train_loader, val_loader, test_loader, train_dataset, val_dataset, test_dataset

Определим класс модели ReXNet150

In [21]:
class ReXNet150Wrapper(nn.Module):
    def __init__(self, pretrained_path, num_classes=2, dropout=0.3, freeze_backbone=False):
        super().__init__()
        self.backbone = timm.create_model('rexnet_150', pretrained=False, num_classes=0, global_pool='')

        if os.path.exists(pretrained_path):
            state_dict = torch.load(pretrained_path, map_location='cpu')

            if isinstance(state_dict, dict):
                if 'model' in state_dict:
                    state_dict = state_dict['model']
                elif 'state_dict' in state_dict:
                    state_dict = state_dict['state_dict']

                cleaned = {}
                for k, v in state_dict.items():
                    k = k.replace('module.', '').replace('backbone.', '').replace('model.', '')
                    cleaned[k] = v
                
                missing, unexpected = self.backbone.load_state_dict(cleaned, strict=False)
                print(f"[ReXNet150] Loaded weights from {pretrained_path}")
                print(f"  Missing keys: {len(missing)}, Unexpected keys: {len(unexpected)}")
            else:
                print(f"[WARNING] Unexpected checkpoint format: {type(state_dict)}")
        else:
            print(f"[WARNING] Pretrained weights not found at {pretrained_path}")

        feat_dim = self.backbone.num_features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(feat_dim, feat_dim // 2),
            nn.ReLU(inplace=True),
            nn.BatchNorm1d(feat_dim // 2),
            nn.Dropout(dropout),
            nn.Linear(feat_dim // 2, num_classes)
        )
        
        if freeze_backbone:
            for param in self.backbone.parameters():
                param.requires_grad = False
            print("[ReXNet150] Backbone frozen")

    def forward(self, x):
        if x.dim() == 5:
            B, T, C, H, W = x.shape
            x = x.view(B * T, C, H, W)
            is_video = True
        else:
            is_video = False

        x = self.backbone(x)  
        x = self.pool(x)      
        x = torch.flatten(x, 1) 
        
        if is_video:
            x = x.view(B, T, -1)
            x = x.mean(dim=1)
        
        x = self.dropout(x)
        return self.classifier(x)


In [22]:
class ReXNetLightning(pl.LightningModule):
    def __init__(self, pretrained_path, num_classes=2, lr=1e-4, weight_decay=1e-4, freeze_backbone=False):
        super().__init__()
        self.save_hyperparameters()
        
        self.model = ReXNet150Wrapper(
            pretrained_path=pretrained_path,
            num_classes=num_classes,
            freeze_backbone=freeze_backbone
        )
        
        self.criterion = nn.CrossEntropyLoss()

        task = 'binary' if num_classes == 2 else 'multiclass'
        self.train_acc = Accuracy(task=task, num_classes=num_classes)
        self.val_acc = Accuracy(task=task, num_classes=num_classes)
        self.val_f1 = F1Score(task=task, num_classes=num_classes)
        self.val_auroc = AUROC(task=task, num_classes=num_classes)
        self.test_acc = Accuracy(task=task, num_classes=num_classes)
        self.test_f1 = F1Score(task=task, num_classes=num_classes)
        self.test_auroc = AUROC(task=task, num_classes=num_classes)

    def forward(self, x):
        return self.model(x)

    def _shared_step(self, batch, batch_idx, stage='train'):
        frames, labels = batch 

        logits = self(frames)
        loss = self.criterion(logits, labels)

        preds = torch.argmax(logits, dim=1)
        
        if stage == 'train':
            self.train_acc(preds, labels)
            self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True, sync_dist=True)
            self.log('train_acc', self.train_acc, on_epoch=True, prog_bar=True, sync_dist=True)
        elif stage == 'val':
            self.val_acc(preds, labels)
            self.val_f1(preds, labels)
            self.val_auroc(logits[:, 1] if logits.shape[1] == 2 else logits, labels)
            self.log('val_loss', loss, on_epoch=True, prog_bar=True, sync_dist=True)
            self.log('val_acc', self.val_acc, on_epoch=True, prog_bar=True, sync_dist=True)
            self.log('val_f1', self.val_f1, on_epoch=True, prog_bar=True, sync_dist=True)
            self.log('val_auroc', self.val_auroc, on_epoch=True, prog_bar=True, sync_dist=True)
        else:  
            self.test_acc(preds, labels)
            self.test_f1(preds, labels)
            self.test_auroc(logits[:, 1] if logits.shape[1] == 2 else logits, labels)
            self.log('test_acc', self.test_acc, on_epoch=True, prog_bar=True, sync_dist=True)
            self.log('test_f1', self.test_f1, on_epoch=True, prog_bar=True, sync_dist=True)
            self.log('test_auroc', self.test_auroc, on_epoch=True, prog_bar=True, sync_dist=True)
        
        return loss

    def training_step(self, batch, batch_idx):
        return self._shared_step(batch, batch_idx, stage='train')

    def validation_step(self, batch, batch_idx):
        return self._shared_step(batch, batch_idx, stage='val')

    def test_step(self, batch, batch_idx):
        return self._shared_step(batch, batch_idx, stage='test')

    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr=self.hparams.lr, weight_decay=self.hparams.weight_decay)
        scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=20, eta_min=1e-6)
        
        return {'optimizer': optimizer, 'lr_scheduler': 
                {'scheduler': scheduler, 'interval': 'epoch', 'frequency': 1}
        }

In [26]:
pretrained_weights = '/kaggle/input/datasets/mariaspasyuk/pretrained-rexnet-enet/state_vggface2_rexnet_150.pt'

num_workers = 2
epochs = 8
lr = 1e-4
num_classes = 2

train_loader, val_loader, test_loader, *_ = create_dataloaders(root_dir='/kaggle/input/datasets/mariaspasyuk', batch_size=16, frames_per_video=16, val_max_real=None, val_max_fake=200)

Found 3380 fake videos in train
Found 3220 real videos in train
 После фильтрации: 3220 real, 3380 fake
 Total 6600 videos for train split
 Real (0): 3220
 Fake (1): 3380
Found 361 fake videos in val
Found 168 real videos in val
 После фильтрации: 168 real, 200 fake
 Total 368 videos for val split
 Real (0): 168
 Fake (1): 200
Found 1251 fake videos in test
Found 600 real videos in test
 После фильтрации: 600 real, 600 fake
 Total 1200 videos for test split
 Real (0): 600
 Fake (1): 600


In [27]:
torch.cuda.empty_cache()
model = ReXNetLightning(pretrained_path=pretrained_weights, num_classes=num_classes, lr=lr, freeze_backbone=True)
    
checkpoint_callback = ModelCheckpoint(dirpath='checkpoints/', filename='rexnet150-{epoch:02d}-{val_auroc:.4f}', monitor='val_auroc', mode='max', save_top_k=3, save_last=True, verbose=True)
early_stopping = EarlyStopping(monitor='val_auroc', mode='max', patience=2, min_delta=0.002, verbose=True)

lr_monitor = LearningRateMonitor(logging_interval='epoch')
logger = TensorBoardLogger('tb_logs/', name='rexnet150_deepfake')

[ReXNet150] Loaded weights from /kaggle/input/datasets/mariaspasyuk/pretrained-rexnet-enet/state_vggface2_rexnet_150.pt
  Missing keys: 0, Unexpected keys: 0
[ReXNet150] Backbone frozen


In [ ]:
trainer = pl.Trainer(
        accelerator='cuda' if torch.cuda.is_available() else 'cpu',
        devices=1,
        max_epochs=15,
        callbacks=[checkpoint_callback, early_stopping, lr_monitor],
        logger=logger,
        log_every_n_steps=10,
        precision=16,
        gradient_clip_val=1.0,
        enable_progress_bar=True
    )
    
trainer.fit(model, train_loader, val_loader)

In [15]:
print(f"Best checkpoint: {checkpoint_callback.best_model_path}")
print(f"Best val AUC: {checkpoint_callback.best_model_score:.4f}")
if len(test_loader.dataset) > 0:
    print("Test evaluation:")
    best_model = ReXNetLightning.load_from_checkpoint(checkpoint_callback.best_model_path)
    trainer.test(best_model, test_loader)


Best checkpoint: /kaggle/working/checkpoints/rexnet150-epoch=05-val_auroc=0.9599.ckpt
Best val AUROC: 0.9599

Running test evaluation:


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


[ReXNet150] Loaded weights from /kaggle/input/datasets/mariaspasyuk/pretrained-rexnet-enet/state_vggface2_rexnet_150.pt
  Missing keys: 0, Unexpected keys: 0
[ReXNet150] Backbone frozen


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │     0.873041570186615     │
│        test_auroc         │    0.9478950500488281     │
│          test_f1          │    0.8985757231712341     │
└───────────────────────────┴───────────────────────────┘

In [ ]:
import os
import shutil
from IPython.display import FileLink

folder_path = '/kaggle/working/checkpoints' 
archive_path = '/kaggle/working/archived_checkpoints'  

shutil.make_archive(archive_path, 'zip', folder_path)

print(f"Архив создан: {archive_path}.zip")
display(FileLink(f"{archive_path}.zip"))

In [ ]:
import os
import shutil
from IPython.display import FileLink

folder_path = '/kaggle/working/tb_logs'
archive_path = '/kaggle/working/archived_tb_logs' 

shutil.make_archive(archive_path, 'zip', folder_path)

print(f"Архив создан: {archive_path}.zip")
display(FileLink(f"{archive_path}.zip"))

# Тестирование на датасете хуавея

In [11]:
import os
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as T
import numpy as np
from pathlib import Path

class HuaweiTestDataset(Dataset):
    def __init__(self, csv_path, frames_root, num_frames=16, transform=None):
        self.df = pd.read_csv(csv_path)
        self.frames_root = Path(frames_root)
        self.num_frames = num_frames
        self.transform = transform
        
        self.video_data = []
        
        for _, row in self.df.iterrows():
            obj_id = str(row['obj_id'])
            label = int(row['label']) if not pd.isna(row['label']) else -1
            
            video_dir = self.frames_root / obj_id
            if video_dir.exists():
                frames = list(video_dir.glob('*.jpg')) + list(video_dir.glob('*.png'))
                frames = sorted(frames, key=lambda x: int(x.stem) if x.stem.isdigit() else 0)
                
                if len(frames) > 0:
                    self.video_data.append({'obj_id': obj_id, 'label': label, 'frames': frames})
        
        print(f"Найдено {len(self.video_data)} видео для обработки.")

    def __len__(self):
        return len(self.video_data)

    def __getitem__(self, idx):
        item = self.video_data[idx]
        frames_paths = item['frames']

        final_frames = []
        
        if len(frames_paths) >= self.num_frames:
            indices = np.linspace(0, len(frames_paths) - 1, self.num_frames, dtype=int)
            selected_paths = [frames_paths[i] for i in indices]
            final_frames = selected_paths
        else:
            final_frames = list(frames_paths)
            padding_needed = self.num_frames - len(final_frames)
            last_frame = final_frames[-1]
            final_frames.extend([last_frame] * padding_needed)

        frames_tensor = []
        for f_path in final_frames:
            img = Image.open(f_path).convert('RGB')
            if self.transform:
                img = self.transform(img)
            frames_tensor.append(img)
            
        frames_tensor = torch.stack(frames_tensor)
        
        return frames_tensor, item['label'] 

def get_test_transforms():
    return T.Compose([
        T.Resize(224),
        T.CenterCrop(224),
        T.ToTensor(),
        T.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
    ])

def create_test_dataloader(
    csv_path='/kaggle/input/datasets/mariaspasyuk/huawei-test/d83a0ce6-dc87-46a6-9679-98a71cf91886.csv',
    frames_root='/kaggle/input/datasets/mariaspasyuk/hweidatasetfaces/hweiDatasetFaces',
    num_frames=16,
    batch_size=8,
    num_workers=2
):
    dataset = HuaweiTestDataset(csv_path=csv_path, frames_root=frames_root, num_frames=num_frames, transform=get_test_transforms())
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, pin_memory=True)
    
    return dataset, dataloader


dataset, dataloader = create_test_dataloader()

Найдено 45 видео для обработки.


10 эпох и замороженный backbone

In [31]:
if len(dataloader.dataset) > 0:
    print("Тестирование на данных Хуавея")
    best_model = ReXNetLightning.load_from_checkpoint(checkpoint_callback.best_model_path)
    trainer.test(best_model, dataloader)


Running test evaluation...


LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


[ReXNet150] Loaded weights from /kaggle/input/datasets/mariaspasyuk/pretrained-rexnet-enet/state_vggface2_rexnet_150.pt
  Missing keys: 0, Unexpected keys: 0
[ReXNet150] Backbone frozen


/usr/local/lib/python3.12/dist-packages/pytorch_lightning/utilities/_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

/usr/local/lib/python3.12/dist-packages/torchmetrics/utilities/prints.py:43: UserWarning: No negative samples in 
targets, false positive value should be meaningless. Returning zero tensor in false positive score
  warnings.warn(*args, **kwargs)

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│         test_acc          │    0.4000000059604645     │
│        test_auroc         │    0.5474138259887695     │
│          test_f1          │    0.3720930218696594     │
└───────────────────────────┴───────────────────────────┘


✅ Training finished!
Best checkpoint: /kaggle/working/checkpoints/rexnet150-epoch=08-val_auroc=0.9719.ckpt
Best val AUROC: 0.9719
